# Exploring MNIST

Before training any model, we look at the data. Run each cell in order with **Shift+Enter**.

In this notebook we:
1. Load MNIST with `torchvision`
2. Look at a grid of 20 handwritten digits
3. Print one image as its raw 28×28 numbers. **An image is just numbers.**
4. See what normalization does to those numbers
5. Check the class balance

The helper functions live in `src/` (and are tested with pytest), so this notebook stays short.

In [ ]:
from src.data import count_labels, get_raw_image, load_datasets
from src.explore import format_image, plot_class_balance, plot_sample_grid

## 1. Load MNIST

`load_datasets()` downloads MNIST into `data/` the first time (about 11 MB), then reuses it.
It returns three datasets:

- **train** (55,000): what the model learns from
- **validation** (5,000): used to compare models and tune settings
- **test** (10,000): used **once**, at the very end, for an honest final score

In [ ]:
train, val, test = load_datasets()
print(f"Train: {len(train):,}  Validation: {len(val):,}  Test: {len(test):,}")

## 2. A grid of 20 sample digits

Each dataset item is a pair `(image, label)`. The label (shown above each image) is the correct answer.

In [ ]:
plot_sample_grid(train, rows=4, cols=5);

## 3. An image is just numbers

Here is the first training image as its raw pixel values:

- `.` = **0**, black background
- **255** = full white ink
- values in between = soft gray edges of the pen stroke

Can you see the digit in the numbers?

In [ ]:
index = 0  # try other values, e.g. 4 or 11
raw = get_raw_image(train, index)
_, label = train[index]

print(f"Label: {label}   Shape: {tuple(raw.shape)}   Total numbers: {raw.numel()}\n")
print(format_image(raw))

**28 × 28 = 784 numbers.** That is *all* a model ever sees: no pen, no paper, just a grid of brightness values.
Every model in this project is a function that turns those 784 numbers into one of 10 digits.

## 4. What normalization does

The models don't get raw 0–255 values. `src/data.py` applies two steps:

1. `ToTensor()`: divides by 255, giving **0.0 to 1.0**
2. `Normalize(0.1307, 0.3081)`: subtracts the dataset's average and divides by its spread, so values center around **0**

Same picture, different scale. Neural networks learn faster with inputs near 0.

In [ ]:
normalized, _ = train[index]

print(f"Raw:        shape {tuple(raw.shape)},     values {raw.min().item()} to {raw.max().item()}")
print(f"Normalized: shape {tuple(normalized.shape)}, values {normalized.min():.2f} to {normalized.max():.2f}")

The normalized image has an extra dimension at the front, `(1, 28, 28)`. That is the **channel**
dimension: 1 for grayscale (a color image would have 3: red, green, blue). The CNN in Phase 4 needs it.

## 5. Class balance

Are some digits much more common than others? If 90% of images were "1", a model that always guessed "1"
would score 90% accuracy and fool us.

In [ ]:
counts = count_labels(train)
for digit, count in enumerate(counts):
    print(f"{digit}: {count:>5,}  ({count / len(train):.1%})")

plot_class_balance(counts);

Every digit is roughly 9–11% of the data, so the classes are **balanced** and plain accuracy is a fair score.

## Takeaways

- MNIST: 70,000 grayscale 28×28 images of digits 0–9
- An image is a grid of 784 numbers; models learn patterns in those numbers
- We normalize pixels to center them around 0
- The classes are balanced, so accuracy is a fair way to compare models

**Next:** Phase 2, a scikit-learn baseline that sees each image as one row of 784 numbers.